# Model 3 - tooth type (YOLO11) -> best.pt

Trains YOLO11 to detect each tooth and label **incisor / canine / premolar / molar**, and
produces a portable **best.pt** (+ ONNX) you can run offline - like Models 1 & 2.

**Setup (pick ONE data source in cell 2):**
- **A. Kaggle dataset (recommended):** upload `Dental_Dataset_Level2_v1i_yolov11.zip` as a Kaggle
  Dataset, then *Add Input* it to this notebook. No API key needed.
- **B. Roboflow pull:** set Kaggle Secret `ROBOFLOW_API_KEY` and flip `USE_ROBOFLOW=True`.

Then: GPU **T4 x2**, Internet **On** -> **Save & Run All** (~30-60 min on T4).
Dataset: keshav-raja/dental_dataset_level2-fazpu v1 (CC BY 4.0). Research prototype, not a diagnosis.

In [ ]:
# 0) settings
MODEL, IMGSZ, EPOCHS, SMOKE = "yolo11s.pt", 640, 100, False
USE_ROBOFLOW = False   # False = use an added Kaggle dataset; True = pull from Roboflow by API key
WORKSPACE, PROJECT, VERSION = "keshav-raja", "dental_dataset_level2-fazpu", 1

In [ ]:
# 1) install
import subprocess, sys, os, glob
subprocess.run([sys.executable,"-m","pip","install","-q","ultralytics>=8.3.0","roboflow"], check=True)
import torch
print("GPUs:", torch.cuda.device_count(), [torch.cuda.get_device_name(i) for i in range(torch.cuda.device_count())])

In [ ]:
# 2) locate the dataset and WRITE our own data.yaml (works whether or not the zip has one)
import os, glob, yaml
DATA_YAML=None
if USE_ROBOFLOW:
    key=""
    if not key:
        try:
            from kaggle_secrets import UserSecretsClient
            key=UserSecretsClient().get_secret("ROBOFLOW_API_KEY")
        except Exception: pass
    assert key, "Set ROBOFLOW_API_KEY, or use a Kaggle dataset (USE_ROBOFLOW=False)."
    from roboflow import Roboflow
    ds=Roboflow(api_key=key).workspace(WORKSPACE).project(PROJECT).version(VERSION).download("yolov11")
    root=ds.location
else:
    # find the folder that contains train/images anywhere under /kaggle/input
    hits=glob.glob("/kaggle/input/**/train/images", recursive=True)
    assert hits, "No train/images found under /kaggle/input - did you Add Input your dataset?"
    root=os.path.dirname(os.path.dirname(hits[0]))   # .../tooth_type
print("dataset root:", root)
d={"path":root,"train":"train/images","val":"valid/images","test":"test/images",
   "nc":4,"names":["canine","incisor","molar","premolar"]}
DATA_YAML="/kaggle/working/data.yaml"; yaml.safe_dump(d, open(DATA_YAML,"w"))
print(open(DATA_YAML).read())

In [ ]:
# 3) train YOLO11 (4 classes: incisor / canine / premolar / molar)
from ultralytics import YOLO
ep = 1 if SMOKE else EPOCHS
m = YOLO(MODEL)
m.train(data=DATA_YAML, epochs=ep, imgsz=IMGSZ, device="0,1", batch=16,
        project="/kaggle/working/runs", name="toothtype", exist_ok=True, patience=25, seed=42)
print("done")

In [ ]:
# 4) evaluate on the test split + per-class table
best="/kaggle/working/runs/toothtype/weights/best.pt"
m = YOLO(best)
mt = m.val(data=DATA_YAML, split="test", imgsz=IMGSZ, device="0")
import json
names = m.names; per={}
try:
    for i,c in enumerate(mt.box.ap_class_index):
        per[names[int(c)]]={"map50":round(float(mt.box.ap50[i]),4),"map50_95":round(float(mt.box.ap[i]),4)}
except Exception as e: print("per-class:", e)
summary={"model":MODEL,"test":{"map50":round(float(mt.box.map50),4),"map50_95":round(float(mt.box.map),4),"perClass":per},"classes":list(names.values())}
open("/kaggle/working/metrics.json","w").write(json.dumps(summary,indent=2)); print(json.dumps(summary,indent=2))

In [ ]:
# 5) export ONNX + package -> Output tab -> model3_toothtype_outputs.zip
import shutil, glob
m.export(format="onnx", imgsz=IMGSZ, opset=12)
pkg="/kaggle/working/pkg"; shutil.rmtree(pkg,ignore_errors=True); os.makedirs(pkg+"/weights",exist_ok=True)
for f in glob.glob("/kaggle/working/runs/toothtype/weights/*"): shutil.copy2(f, pkg+"/weights/")
shutil.copy2("/kaggle/working/metrics.json", pkg+"/metrics.json")
for f in glob.glob("/kaggle/working/runs/toothtype/*.png"): shutil.copy2(f, pkg+"/")
shutil.make_archive("/kaggle/working/model3_toothtype_outputs","zip",pkg)
print("Download: Output tab -> model3_toothtype_outputs.zip"); print("weights:", os.listdir(pkg+"/weights"))

In [ ]:
# 6) quick visual check
import glob, matplotlib.pyplot as plt, cv2, os
root=os.path.dirname(DATA_YAML) if not DATA_YAML.startswith("/kaggle/working") else __import__("yaml").safe_load(open(DATA_YAML))["path"]
tst=glob.glob(os.path.join(root,"test","images","*.jpg"))[:1]
if tst:
    r=m.predict(tst[0], conf=0.4, imgsz=IMGSZ, save=False)[0]
    plt.figure(figsize=(9,7)); plt.imshow(cv2.cvtColor(r.plot(),cv2.COLOR_BGR2RGB)); plt.axis("off"); plt.title("Model 3"); plt.show()
    print("teeth:", 0 if r.boxes is None else len(r.boxes))